In [1]:
import torch
import numpy as np


# ---------- Load PH-based results ----------
def load_results(dirname, dname, gnn, diagram_type,
                 fb_one=False, no_ofst=False,
                 extended_persistence=False, extended_persistence2=False,
                 seed=2):

    fp = f"{dirname}/{dname}/2/8/16_hidden/64_outdim/True_dim1/{diagram_type}_{gnn}_{seed}"

    if diagram_type == "forward_backward":
        if fb_one:
            fp += "_fbone1"
        if extended_persistence:
            fp += "_extpers1"
        if extended_persistence2:
            fp += "_extpers21"
        if no_ofst:
            fp += "_noofst1"

    return torch.load(f"{fp}.results")


# ---------- Load PersLay special result ----------
def load_perslay(dname, seed=2):
    fp = f"./results_perslay/{dname}/perslay_{seed}.results"
    return torch.load(fp)


# ---------- Extract ES-max / max ----------
def extract_value(res, mode):
    test = res["test_accuracies"]
    val_best = res["val_accuracies"].argmax()
    return float(test[val_best]) if mode == "es_max" else float(test.max())


# ---------- Build table (3 columns only) ----------
def make_extended_table(dirname, gnn, mode="es_max", seed=2):

    datasets = ["NCI109", "PROTEINS", "IMDB-BINARY", "NCI1"]

    # Extended Persistence = ep2 + no_ofst
    ep2_noofst = ("ep2+noofst", False, True, False, True)

    # Ours = 3 variants
    ours_variants = [
        ("fb_one+noofst", True, True, False, False),
        ("fb_one_only",   True, False, False, False),
        ("plain_fb",      False, False, False, False),
    ]

    table = []

    for d in datasets:
        row = {}

        # ----- Extended Persistence: ep2 + no_ofst -----
        try:
            name, fb1, no_ofst, ep, ep2 = ep2_noofst
            res = load_results(dirname, d, gnn, "forward_backward",
                               fb_one=fb1, no_ofst=no_ofst,
                               extended_persistence=ep,
                               extended_persistence2=ep2,
                               seed=seed)
            row["extended"] = extract_value(res, mode)
        except FileNotFoundError:
            row["extended"] = "N/A"

        # ----- PersLay -----
        try:
            res_p = load_perslay(d, seed)
            row["perslay"] = extract_value(res_p, mode)
        except FileNotFoundError:
            row["perslay"] = "N/A"

        # ----- Ours = max among the 3 ours variants -----
        candidate_vals = []
        for name, fb1, no_ofst, ep, ep2 in ours_variants:
            try:
                res = load_results(dirname, d, gnn, "forward_backward",
                                   fb_one=fb1, no_ofst=no_ofst,
                                   extended_persistence=ep,
                                   extended_persistence2=ep2,
                                   seed=seed)
                candidate_vals.append(extract_value(res, mode))
            except FileNotFoundError:
                pass

        row["ours"] = max(candidate_vals) if candidate_vals else "N/A"

        table.append((d, row))

    return table


# ---------- Print table ----------
def print_extended_table(table, gnn, mode):

    headers = ["Extended Persistence", "PersLay", "Ours"]
    col_width = 22

    print(f"\n=== Extended Results for {gnn.upper()} ({mode}) ===")
    print(f"{'Dataset':<{col_width}} " +
          " | ".join(f"{h:<{col_width}}" for h in headers))
    print("-" * (col_width * (len(headers) + 1) + 10))

    for dname, row in table:
        vals = []
        for key in ["extended", "perslay", "ours"]:
            v = row[key]
            if isinstance(v, float):
                vals.append(f"{v:.4f}")
            else:
                vals.append("N/A")

        print(f"{dname:<{col_width}} " +
              " | ".join(f"{v:<{col_width}}" for v in vals))


In [2]:
dirname = "results/main"
gnn = "gin"
mode = "es_max"
print_extended_table(make_extended_table(dirname, gnn, mode, seed=2), gnn, mode)


=== Extended Results for GIN (es_max) ===
Dataset                Extended Persistence   | PersLay                | Ours                  
--------------------------------------------------------------------------------------------------
NCI109                 0.7821                 | 0.6828                 | 0.7821                
PROTEINS               0.7411                 | 0.6607                 | 0.7321                
IMDB-BINARY            0.6300                 | 0.7000                 | 0.7300                
NCI1                   0.7859                 | 0.6886                 | 0.8151                


In [3]:
dirname = "results/main"
gnn = "gcn"
mode = "es_max"
print_extended_table(make_extended_table(dirname, gnn, mode, seed=2), gnn, mode)


=== Extended Results for GCN (es_max) ===
Dataset                Extended Persistence   | PersLay                | Ours                  
--------------------------------------------------------------------------------------------------
NCI109                 0.7748                 | 0.6828                 | 0.7748                
PROTEINS               0.7232                 | 0.6607                 | 0.7232                
IMDB-BINARY            0.6600                 | 0.7000                 | 0.7000                
NCI1                   0.8005                 | 0.6886                 | 0.8078                
